# 💥 BlastRadius — A100 Training Notebook (v2 — Hackathon Ready)

> **Run every cell top-to-bottom. Each stage validates before moving to the next.**
>
> **Timeline estimate on A100 80GB:**
> - Cell 1: Setup ~3-5 min
> - Cell 2: SFT data generation — **SKIPPED** (pre-generated data included)
> - Cell 3: SFT training ~15-20 min (Qwen2.5-7B-Instruct 4-bit)
> - Cell 4: Validate SFT ~1 min
> - Cell 5: GRPO RL training ~2-4 hours (WandB tracked, SIGTERM-safe)
> - Cell 6: Validate GRPO ~1 min
> - Cell 7: Push to HF Hub ~5 min
> - Cell 8: Benchmark baseline ~3 min
>
> **Total: ~3-5 hours**
>
> GitHub: https://github.com/Divyansh-9/BlastRadius
> Live Space: https://huggingface.co/spaces/Idred/BlastRadius-OpenEnv

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Environment Setup
# Clones from GitHub (development branch), installs all deps
# ─────────────────────────────────────────────────────────────
import os

# Verify GPU is available
!nvidia-smi

# Clone from the development branch (latest hardened code)
REPO_URL = "https://github.com/Divyansh-9/BlastRadius.git"
BRANCH   = "development"

!git clone --branch {BRANCH} {REPO_URL} blastradius
%cd blastradius

# Install core dependencies
!pip install -e '.[train]' -q

# Unsloth — pinned for GRPO + vLLM colocation compatibility
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' -q
!pip install 'trl>=0.9.0' wandb huggingface_hub -q

# Create output dirs
!mkdir -p sft_data models

print('\n✅ Setup complete. GPU ready for training.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — SFT Data Generation (SKIP IF DATA ALREADY EXISTS)
# Pre-generated expert_trajectories.jsonl is committed to the
# repo in sft_data/. Only run this cell if you want fresh data.
# ─────────────────────────────────────────────────────────────
import os

SKIP_GENERATION = os.path.exists('sft_data/expert_trajectories.jsonl')

if SKIP_GENERATION:
    import subprocess
    result = subprocess.run(['wc', '-l', 'sft_data/expert_trajectories.jsonl'],
                            capture_output=True, text=True)
    # Windows fallback
    try:
        with open('sft_data/expert_trajectories.jsonl') as f:
            line_count = sum(1 for _ in f)
        print(f'✅ Pre-generated SFT data found: {line_count} training examples')
        print('   Skipping generation — proceeding to Cell 3.')
    except Exception:
        print('✅ sft_data/expert_trajectories.jsonl exists — skipping generation')
else:
    print('No SFT data found — generating now...')
    # ⚠️  Requires an OpenAI-compatible teacher API key
    os.environ['TEACHER_API_KEY'] = 'sk-...'       # ← Replace with your key
    os.environ['TEACHER_API_BASE'] = 'https://integrate.api.nvidia.com/v1'
    os.environ['TEACHER_MODEL'] = 'meta/llama-3.1-8b-instruct'

    !python -m agent.generate_sft_data \
        --episodes 100 \
        --tasks easy medium hard \
        --output sft_data

    print('\n✅ SFT data generation complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Stage 1: Cold-Start SFT Training
# ~15-20 min on A100 80GB
# Model: Qwen2.5-7B-Instruct 4-bit (~4.5GB VRAM)
# Teaches the model: MATPO tag format + SRE domain vocabulary
# ─────────────────────────────────────────────────────────────

# Verify data exists before proceeding
import os
assert os.path.exists('sft_data/expert_trajectories.jsonl'), \
    'ERROR: No SFT data found! Run Cell 2 first.'

!python -m agent.train_sft \
    --model 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' \
    --data  sft_data/expert_trajectories.jsonl \
    --output models/sft_checkpoint

print('\n✅ SFT training complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Validate SFT Checkpoint
# CRITICAL: Do NOT proceed to GRPO if this fails.
# ─────────────────────────────────────────────────────────────
!python -m agent.validate_save --model models/sft_checkpoint

# ⛔ If this cell fails:
#    1. Check disk space: !df -h
#    2. Re-run Cell 3
#    3. Check for CUDA OOM in Cell 3 output

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Stage 2: GRPO Reinforcement Learning
#
# SPOT-INSTANCE SAFE:
#   - SIGTERM hook saves emergency checkpoint to Hub on preemption
#   - Wall-clock alarm (2h default) prevents runaway credit drain
#   - hub_strategy=checkpoint pushes async every 200 steps
#   - resume_from_checkpoint auto-detects trainer_state.json
#
# MEMORY PROFILE (A100 80GB, hardware-profile=a100):
#   - vLLM pool:      56GB  (0.70 × 80GB)
#   - LoRA + grad:    ~16GB
#   - Safety margin:  ~8GB
#   - Peak:           ~72GB  ✅ fits
#
# HYPERPARAMETERS (hardened):
#   - learning_rate=1e-6  (stable for Qwen2.5, prevents divergence)
#   - beta=0.1            (strong KL constraint for short 2-epoch runs)
#   - max_seq_length=2048 (handles verbose hard-scenario observations)
#   - num_generations=16  (A100 headroom allows full rollout diversity)
# ─────────────────────────────────────────────────────────────
import os

# ── Credentials (set as HF Job secrets or fill below) ──────
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')   # Set in HF Job secrets
WANDB_ENTITY  = os.environ.get('WANDB_ENTITY', 'blastradius')
HUB_MODEL_ID  = os.environ.get('HUB_MODEL_ID', 'Divyansh-9/BlastRadius-GRPO-Checkpoints')
HF_TOKEN      = os.environ.get('HF_TOKEN', '')

os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['HF_TOKEN']      = HF_TOKEN

# ── Validate checkpoint exists ──────────────────────────────
assert os.path.exists('models/sft_checkpoint'), \
    'ERROR: SFT checkpoint not found! Run Cells 3 & 4 first.'

# ── Launch GRPO ─────────────────────────────────────────────
!python -m agent.train_grpo \
    --model   models/sft_checkpoint \
    --data    sft_data/expert_trajectories.jsonl \
    --output  models/grpo_checkpoint \
    --hardware-profile a100 \
    --wandb-project   blastradius-grpo \
    --wandb-entity    {WANDB_ENTITY} \
    --hub-model-id    {HUB_MODEL_ID} \
    --max-runtime-hours 4.0

# ── What to watch in WandB ──────────────────────────────────
# reward/format_reward_func      → target: ↑ toward 0.75+
# reward/environment_reward_func → key RL signal, watch for +trend
# reward                         → overall mean, should rise steadily
# kl                             → should stay < 0.5 (KL constraint working)

print('\n✅ GRPO training complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Validate GRPO Checkpoint
# ─────────────────────────────────────────────────────────────
import os

# Fall back to SFT checkpoint if GRPO failed
BEST_MODEL = 'models/grpo_checkpoint' \
    if os.path.exists('models/grpo_checkpoint/trainer_state.json') \
    else 'models/sft_checkpoint'

print(f'Using model: {BEST_MODEL}')
!python -m agent.validate_save --model {BEST_MODEL}

# ⛔ If GRPO checkpoint is corrupt, proceed with SFT checkpoint.
# A working SFT model scores better than a corrupt GRPO model.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — Push Best Model to HuggingFace Hub
# ─────────────────────────────────────────────────────────────
from huggingface_hub import HfApi
import os

HF_TOKEN = os.environ.get('HF_TOKEN', '')
HF_REPO  = os.environ.get('HF_REPO', 'Divyansh-9/BlastRadius-GRPO')

# Use best available checkpoint
BEST_MODEL = 'models/grpo_checkpoint' \
    if os.path.exists('models/grpo_checkpoint/trainer_state.json') \
    else 'models/sft_checkpoint'

print(f'Pushing {BEST_MODEL} → {HF_REPO} ...')

api = HfApi()
api.create_repo(repo_id=HF_REPO, repo_type='model',
                token=HF_TOKEN, exist_ok=True)
api.upload_folder(
    folder_path=BEST_MODEL,
    repo_id=HF_REPO,
    repo_type='model',
    token=HF_TOKEN,
    commit_message=f'BlastRadius GRPO checkpoint — hackathon submission',
)

print(f'\n✅ Model pushed to https://huggingface.co/{HF_REPO}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — Benchmark: Random Baseline vs Trained Model
# Generates the before/after numbers for the pitch deck.
# Runs against all 3 difficulty tiers.
# ─────────────────────────────────────────────────────────────
import sys, random
sys.path.insert(0, '.')

from incident_env.server.incident_environment import IncidentEnvironment
from incident_env.models import IncidentAction

VALID_COMMANDS = [
    'check_status', 'check_logs', 'check_metrics',
    'check_dependencies', 'diagnose',
    'restart_service', 'rollback_deploy', 'scale_service'
]

def score_random_policy(task_id='easy', steps=10):
    """Random policy baseline — no model, just random valid commands."""
    env = IncidentEnvironment()
    env.reset(task_id=task_id)
    total = 0.0
    for _ in range(steps):
        cmd = random.choice(VALID_COMMANDS)
        result = env.step(IncidentAction(command=cmd))
        total += result.get('reward', 0.0)
        if result.get('done', False):
            break
    return total

print('Running 3 episodes per difficulty...')
results = {}
for difficulty in ['easy', 'medium', 'hard']:
    scores = [score_random_policy(difficulty) for _ in range(3)]
    results[difficulty] = sum(scores) / len(scores)
    print(f'  [{difficulty:6}] random policy mean reward: {results[difficulty]:.4f}')

print()
print('─' * 50)
print('These are your BASELINE numbers (random policy).')
print('After GRPO training, run agent/benchmark.py to get')
print('trained model scores and compare for your pitch slide.')
print()
print('Command:')
print('  python agent/benchmark.py --episodes 3')
print('  # → Generates docs/runs/benchmark_<timestamp>.html')